In [ ]:
#!pip install category_encoders
#!pip install kagglehub
#!pip install matplotlib
#!pip install seaborn
#!pip install hvplot holoviews bokeh panel param
#!pip install numpy

In [ ]:
import kagglehub
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import category_encoders as ce
import hvplot.pandas
import holoviews as hv
import math
hv.extension('bokeh') 
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score, roc_curve
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score

In [ ]:
# Download latest version
path = kagglehub.dataset_download("pavansubhasht/ibm-hr-analytics-attrition-dataset")

print("Path to dataset files:", path)

# Assuming 'aug_train.csv' is the primary data file within the dataset directory
perf_path = 'WA_Fn-UseC_-HR-Employee-Attrition.csv'
full_csv_path = os.path.join(path, perf_path)

# Check if the file exists before attempting to read
if os.path.exists(full_csv_path):
    df = pd.read_csv(full_csv_path)
    print(f"Successfully loaded {perf_path} into a DataFrame.")
    display(df.head())
else:
    print(f"Error: The file '{perf_path}' was not found in '{path}'.")
    print("Available files in the directory:", os.listdir(path))

# New Section

In [ ]:
df.shape

In [ ]:
df.isna().sum()

In [ ]:
df.columns

In [ ]:
237/1233

In [ ]:
df['Attrition'].value_counts()

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
sns.countplot(x='Attrition', data=df)

In [ ]:
for i in df.columns:
  print(i)

In [ ]:
#Number of unique values per column

lista={}
for i in df.columns:
  #lista[i] = df[i].nunique()
  print(i, ':', df[i].nunique())

In [ ]:
df[['Over18', 'StandardHours', 'EmployeeCount']].value_counts()

In [ ]:
# Fix: Drop the 'Over18' column by name and specify axis=1
#f = df.drop('Over18', axis=1)
#df = df.drop('StandardHours', axis=1)
#df = df.drop('EmployeeCount', axis=1)
#df = df.drop('EmployeeNumber', axis=1)
df = df.drop(['EmployeeCount', 'StandardHours', 'Over18', 'EmployeeNumber'], axis=1)

In [ ]:
attrition_palette = {'No': 'steelblue', 'Yes': 'red'}

plot_columns = [col for col in df.columns if len(df[col].unique()) < 21 and col != 'Attrition']

num_plots = len(plot_columns)
num_cols = 4
num_rows = (num_plots + num_cols - 1) // num_cols

fig, axes = plt.subplots(num_rows, num_cols, figsize=(num_cols * 4, num_rows * 3.5))
axes = axes.flatten()

for i, col in enumerate(plot_columns):
    sns.countplot(x=col, hue='Attrition', data=df, ax=axes[i], palette=attrition_palette)
    axes[i].set_title(col)
    axes[i].tick_params(axis='x', rotation=75)

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
#All non-integer columns

print(df.dtypes[df.dtypes == 'object'])

In [ ]:
df['Attrition'] = df['Attrition'].apply(lambda x: 1 if x=='Yes' else 0)
df['Gender'] = df['Gender'].apply(lambda x: 1 if x=='Male' else 0)
df['OverTime'] = df['OverTime'].apply(lambda x: 1 if x=='Yes' else 0)

In [ ]:
df[(df.dtypes[df.dtypes == 'int64']).keys().to_numpy()]

In [ ]:
sns.heatmap(df[(df.dtypes[df.dtypes == 'int64']).keys().to_numpy()].corr())

In [ ]:
#sns.boxplot(x='Attrition', y='OverTime', data=df)
df.groupby('Attrition')['OverTime'].value_counts().plot(kind='bar')

In [ ]:
sns.boxplot(x='Attrition', y='JobLevel', data=df)

In [ ]:
df.columns

In [ ]:
df.groupby('Attrition')['MonthlyIncome'].describe()

In [ ]:
sns.boxplot(x='Attrition', y='MonthlyIncome', data=df)
plt.title('Attrition vs. MonthlyIncome')

In [ ]:
sns.boxplot(x='Attrition', y='YearsAtCompany', data=df)

In [ ]:
sns.boxplot(x='Attrition', y='YearsInCurrentRole', data=df)

In [ ]:
sns.boxplot(x='Attrition', y='YearsSinceLastPromotion', data=df)

In [ ]:
sns.boxplot(x='Attrition', y='YearsWithCurrManager', data=df)

In [ ]:
df.groupby('Attrition')['TotalWorkingYears'].describe()

In [ ]:
sns.boxplot(x='Attrition', y='TotalWorkingYears', data=df)
plt.title('Attrition vs. TotalWorkingYears')

In [ ]:
sns.boxplot(x='Attrition', y='DistanceFromHome', data=df)
plt.title('Attrition vs. DistanceFromHome')

In [ ]:
df[(df.dtypes[df.dtypes == 'int64']).keys().to_numpy()]

In [ ]:
#categorical_cols = ['BusinessTravel', 'Department', 'EducationField', 'JobRole', 'OverTime', 'MaritalStatus']
numerical_cols = df.dtypes[df.dtypes == 'int64'].index.to_list()
categorical_cols = df.dtypes[df.dtypes == 'object'].index.to_list()
categorical_cols

In [ ]:
# Separate features (X) and target (y)
X = df.drop('Attrition', axis=1)
y = df['Attrition']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
# validation split
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.20, stratify=y_train)

#categorical_cols = ['BusinessTravel', 'Department', 'EducationField', 'JobRole', 'OverTime', 'MaritalStatus']
#OneHotEncoder
ohe = ce.OneHotEncoder(cols=categorical_cols)
X_train_encoded = ohe.fit_transform(X_train)
X_test_encoded = ohe.transform(X_test)
X_val_encoded = ohe.fit_transform(X_val)

#Scaling the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_encoded)
X_test_scaled = scaler.transform(X_test_encoded)
X_val_scaled = scaler.fit_transform(X_val_encoded)

In [ ]:
#LogRegression

#Model Def
lr = LogisticRegression()
lr.fit(X_train_scaled, y_train)

#Predict and Score
y_pred_lr = lr.predict(X_val_scaled)
f1_lr = f1_score(y_val, y_pred_lr)
accuracy_lr = accuracy_score(y_val, y_pred_lr)
recall_lr = recall_score(y_val, y_pred_lr)
precision_lr = precision_score(y_val, y_pred_lr)
roc_lr = roc_auc_score(y_val, y_pred_lr)

print(f1_lr , ',' , accuracy_lr , ',' , recall_lr , ',' , precision_lr , ',' , roc_lr)


In [ ]:
def plot_confusion_matrix(cm):
    # Plotting the confusion matrix
    plt.figure(figsize=(6,4));
    sns.heatmap(cm, annot=True, fmt='g', cmap='Blues', cbar=False);  # fmt='g' is for integer formatting
    plt.xlabel('Predicted Labels');
    plt.ylabel('True Labels');
    plt.title('Confusion Matrix');
    plt.show();


cm = confusion_matrix(y_val, y_pred_lr)

plot_confusion_matrix(cm)

In [ ]:
#RandomForest

#Model Def
rf = RandomForestClassifier()
rf.fit(X_train_encoded, y_train)

#Predict and Score
y_pred_rf = rf.predict(X_val_encoded)
f1_rf = f1_score(y_val, y_pred_rf)
accuracy_rf = accuracy_score(y_val, y_pred_rf)
recall_rf = recall_score(y_val, y_pred_rf)
precision_rf = precision_score(y_val, y_pred_rf)
roc_rf = roc_auc_score(y_val, y_pred_rf)

print(f1_rf , ',' , accuracy_rf , ',' , recall_rf , ',' , precision_rf , ',' , roc_rf)

In [ ]:
def plot_confusion_matrix(cm):
    # Plotting the confusion matrix
    plt.figure(figsize=(6,4));
    sns.heatmap(cm, annot=True, fmt='g', cmap='Blues', cbar=False);  # fmt='g' is for integer formatting
    plt.xlabel('Predicted Labels');
    plt.ylabel('True Labels');
    plt.title('Confusion Matrix');
    plt.show();


cm = confusion_matrix(y_val, y_pred_rf)

plot_confusion_matrix(cm)

In [ ]:
df[numerical_cols].min()

In [ ]:
df[numerical_cols].max()

In [ ]:
aux_vec = ['StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear',
'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole',
'YearsSinceLastPromotion', 'YearsWithCurrManager']
print('Min:', '\n',  df[aux_vec].min(), '\n', 'Max:' , '\n' , df[aux_vec].max(),)


In [ ]:
aux_monthrate = np.linspace(df['MonthlyRate'].min(), df['MonthlyRate'].max(), 8)
#intervals = df['MonthlyRate'].value_counts(bins=aux_monthrate).sort_index()
attrition_monthlyrate = df.groupby(pd.cut(df['MonthlyRate'], bins = aux_monthrate, include_lowest=True))['Attrition'].mean()*100
print(attrition_monthlyrate.sort_values(ascending=False))
df.hvplot.hist(y='MonthlyRate', by='Attrition', subplots=False, height=300, bins= aux_monthrate)

In [ ]:
cols_aux= 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager'
plots= []

for col in cols_aux:
    bins_aux = np.linspace(df[col].min(), df[col].max(), 8)
    step = np.ceil((df[col].max() - df[col].min()) / 6)
    bins_aux = np.arange(df[col].min(), df[col].max() + step, step)
    attrition = df.groupby(pd.cut(df[col], bins = bins_aux, include_lowest=True))['Attrition'].mean()*100
    graph = df.hvplot.hist(y=col, by='Attrition', subplots=False, height=300, bins= bins_aux, title=col)
    plots.append(graph)

all_plots = hv.Layout(plots).cols(len(cols_aux))
all_plots


In [ ]:
numerical_summary = df[numerical_cols].apply(lambda x: {
    'uni': x.nunique(),
    'tp': x.dtype,
    'mx': x.max(),
    'mn': x.min()
})
numerical_summary

In [ ]:
categorical_summary = df[categorical_cols].apply(lambda x: {
    'uni': x.nunique(),
    'tp': x.dtype
})
categorical_summary

In [ ]:
df.columns

In [ ]:
df[['HourlyRate', 'DailyRate', 'MonthlyRate', 'MonthlyIncome']].describe()

In [ ]:
Hrate = 'HourlyRate'
Drate = 'DailyRate'
Mrate = 'MonthlyRate'
Minc = 'MonthlyIncome'
pairs = [(Hrate, Drate), (Hrate, Mrate), (Hrate, Minc), (Drate, Mrate), (Drate, Minc), (Mrate, Minc)]

print("Pay Rates & Income Correlations:")
print("-----------------------------------------\n")

for i in pairs:
    print(i)
    print(df[list(i)].corr().iloc[0,1]) 

In [ ]:
df[['JobSatisfaction', 'EnvironmentSatisfaction', 'RelationshipSatisfaction']].describe()

In [ ]:
JSat = 'JobSatisfaction'
ESat = 'EnvironmentSatisfaction'
RSat = 'RelationshipSatisfaction'
pairs = [(JSat, ESat), (JSat, RSat), (RSat, ESat)]

print("Satisfaction Metrics' Correlations:")
print("-----------------------------------------\n")

for i in pairs:
    print(i)
    print(df[list(i)].corr().iloc[0,1]) 

In [ ]:
df['new1'] = (df['DailyRate']/df['HourlyRate']) #Supposed hours needed in a day
df['new1'].sort_values().reset_index(drop=True).plot()


In [ ]:
df['new2'] = (df['MonthlyRate']/df['DailyRate']) #Supposed days needed in a month
df['new2'].sort_values().reset_index(drop=True).plot()

In [ ]:
df['new3'] = (df['MonthlyRate'] - df['MonthlyIncome']) #Diff between Monthly Rate and Monthly Income
df['new3'].sort_values().reset_index(drop=True).plot()

In [ ]:
df.groupby('Attrition')['Education'].value_counts()

In [ ]:
df['PercentSalaryHike'].value_counts()

In [ ]:
df[df['Gender']==1]

In [ ]:
aux_var = df.groupby('Gender')['MonthlyIncome']
aux_var.mean().plot(kind='bar')

In [ ]:
aux_var = df.groupby('Age')['MonthlyIncome']
ax = aux_var.mean().plot()
ax.set_title('Age vs. Income')
ax.set_ylabel('Income')

In [ ]:
aux_var = df.groupby('PercentSalaryHike')['Attrition']
ax = aux_var.value_counts().plot(kind='bar')
ax.set_title('Income for each Education Field')
ax.set_ylabel('Income')

In [ ]:
aux_var = df.groupby('Education')['MonthlyIncome']
ax = aux_var.mean().plot(kind='bar')
ax.set_title('Income for each Education Field')
ax.set_ylabel('Income')

In [ ]:
df['JobLevel'].unique()

In [ ]:
df.columns

In [ ]:
aux_inc = df.groupby('EducationField')['MonthlyIncome'].mean()
aux_inv = df.groupby('EducationField')['JobInvolvement'].mean()
aux_lev = df.groupby('EducationField')['JobLevel'].mean()
aux_prf = df.groupby('EducationField')['PerformanceRating'].mean()

fig, ax = plt.subplots(2, 2, figsize=(15, 10))
aux_inc.plot(kind='bar', ax=ax[0, 0])
ax[0, 0].set_title('Income for each Education Field')
ax[0, 0].set_ylabel('Income')
aux_inv.plot(kind='bar', ax=ax[0, 1])
ax[0, 1].set_title('Job Involvement for each Education Field')
ax[0, 1].set_ylabel('Job Involvement')
aux_lev.plot(kind='bar', ax=ax[1, 0])
ax[1, 0].set_title('Job Level for each Education Field')
ax[1, 0].set_ylabel('Job Level')
aux_prf.plot(kind='bar', ax=ax[1, 1])
ax[1, 1].set_title('Performance Rating for each Education Field')
ax[1, 1].set_ylabel('Performance Rating')

for a in ax.flatten():
    a.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


In [ ]:
aux_inc = df.groupby('WorkLifeBalance')['MonthlyIncome'].mean()
aux_inv = df.groupby('WorkLifeBalance')['JobInvolvement'].mean()
aux_lev = df.groupby('WorkLifeBalance')['JobLevel'].mean()
aux_prf = df.groupby('WorkLifeBalance')['PerformanceRating'].mean()

fig, ax = plt.subplots(2, 2, figsize=(15, 10))
aux_inc.plot(kind='bar', ax=ax[0, 0])
ax[0, 0].set_title('Income for each WorkLifeBalance')
ax[0, 0].set_ylabel('Income')
aux_inv.plot(kind='bar', ax=ax[0, 1])
ax[0, 1].set_title('Job Involvement for each WorkLifeBalance')
ax[0, 1].set_ylabel('Job Involvement')
aux_lev.plot(kind='bar', ax=ax[1, 0])
ax[1, 0].set_title('Job Level for each WorkLifeBalance')
ax[1, 0].set_ylabel('Job Level')
aux_prf.plot(kind='bar', ax=ax[1, 1])
ax[1, 1].set_title('Performance Rating for each WorkLifeBalance')
ax[1, 1].set_ylabel('Performance Rating')

for a in ax.flatten():
    a.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


In [ ]:
aux_MKT = df[df['EducationField']=='Marketing'].groupby('Education')['MonthlyIncome'].mean()
aux_HR = df[df['EducationField']=='Human Resources'].groupby('Education')['MonthlyIncome'].mean()
aux_LS = df[df['EducationField']=='Life Sciences'].groupby('Education')['MonthlyIncome'].mean()
aux_MED = df[df['EducationField']=='Medical'].groupby('Education')['MonthlyIncome'].mean()
aux_OTH = df[df['EducationField']=='Other'].groupby('Education')['MonthlyIncome'].mean()
aux_TCH = df[df['EducationField']=='Technical Degree'].groupby('Education')['MonthlyIncome'].mean()
ax = pd.DataFrame({'Marketing' : aux_MKT, 'HR' : aux_HR, 'Life Sciences' : aux_LS, 'Medical' : aux_MED, 'Other' : aux_OTH, 'Technical Degree' : aux_TCH}).plot()
ax.set_title('Income for each Education Level per Department')
ax.set_ylabel('Income')

In [ ]:
aux_var.mean().plot()
plt.xticks(rotation=45, ha='right')
aux_var.mean().plot()
plt.xticks(rotation=45, ha='right')

In [ ]:
aux_var = df.groupby('MaritalStatus')['MonthlyIncome']
aux_var.mean().plot(kind='bar')

In [ ]:
(
    df
    .groupby('BusinessTravel')['Attrition']
    .value_counts(normalize=True)
    .unstack()
    .plot(kind='bar', stacked=True)
)

plt.ylabel('Attrition Proportion')
plt.legend(title='Attrition')
plt.title('BusinessTravel vs. Attrition Proportion')

In [ ]:
df.groupby('Attrition')['BusinessTravel'].value_counts()

In [ ]:
df.groupby('Attrition')['OverTime'].value_counts()

In [ ]:
df.groupby('Attrition')['MaritalStatus'].value_counts()

In [ ]:
(
    df
    .groupby('OverTime')['Attrition']
    .value_counts(normalize=True)
    .unstack()
    .plot(kind='bar', stacked=True)
)

plt.ylabel('Attrition Proportion')
plt.legend(title='Attrition')
plt.title('OverTime vs. Attrition Proportion')

In [ ]:
(
    df
    .groupby('MaritalStatus')['Attrition']
    .value_counts(normalize=True)
    .unstack()
    .plot(kind='bar', stacked=True)
)

plt.ylabel('Attrition Proportion')
plt.legend(title='Attrition')
plt.title('Marital Status vs. Attrition Proportion')


In [ ]:
(
    df
    .groupby('Gender')['Attrition']
    .value_counts(normalize=True)
    .unstack()
    .plot(kind='bar', stacked=True)
)

plt.ylabel('Proportion')
plt.legend(title='Attrition')


In [ ]:
(
    df
    .groupby('Education')['Attrition']
    .value_counts(normalize=True)
    .unstack()
    .plot(kind='bar', stacked=True)
)

plt.ylabel('Proportion')
plt.legend(title='Attrition')


In [ ]:
df.columns

In [ ]:
(
    df
    .groupby('PerformanceRating')['Attrition']
    .value_counts(normalize=True)
    .unstack()
    .plot(kind='bar', stacked=True)
)

plt.ylabel('Proportion')
plt.legend(title='Attrition')

In [ ]:
(
    df
    .groupby('TotalWorkingYears')['Attrition']
    .value_counts(normalize=True)
    .unstack()
    .plot(kind='bar', stacked=True)
)

plt.ylabel('Proportion')
plt.legend(title='Attrition')

In [ ]:
(
    df
    .groupby('EducationField')['Attrition']
    .value_counts(normalize=True)
    .unstack()
    .plot(kind='bar', stacked=True)
)

plt.ylabel('Proportion')
plt.legend(title='Attrition')


In [ ]:
aux_categorical_cols = categorical_cols
aux_categorical_cols.append('Gender')

n_cols = 4
n_plots = len(aux_categorical_cols)
n_rows = math.ceil(n_plots / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 5 * n_rows))
axes = axes.flatten()

def plot_attrition_by_category(df, col, ax):
    (
        df
        .groupby(col)['Attrition']
        .value_counts(normalize=True)
        .unstack()
        .plot(kind='bar', stacked=True, ax=ax, legend=False)
    )
    ax.set_title(col)
    ax.set_ylabel('Proportion')
    ax.tick_params(axis='x', rotation=45)
for i, col in enumerate(aux_categorical_cols):
    plot_attrition_by_category(df, col, axes[i])
#plot_attrition_by_category(df, 'Gender', axes[i])
# Remove unused axes
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

# One legend for everything
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, title='Attrition', loc='upper right')

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

def plot_attrition_by_category(df, col):
    (
        df
        .groupby(col)['Attrition']
        .value_counts(normalize=True)
        .unstack()
        .plot(kind='bar', stacked=True, figsize=(6,4))
    )
    plt.ylabel('Attrition Proportion')
    plt.title(col, 'vs Attrition')
    plt.legend(title='Attrition')
    plt.xticks(rotation=45, ha='right')
    
    
for col in categorical_cols:
    plot_attrition_by_category(df, col)
plt.tight_layout()
plt.show()

In [ ]:
'TotalWorkingYears',

In [ ]:
aux_list = ['YearsAtCompany','YearsInCurrentRole','YearsSinceLastPromotion',
            'YearsWithCurrManager','JobLevel', 'OverTime', 'MaritalStatus', 'JobRole', 'EducationField', 'BusinessTravel']

n_cols = 3
n_plots = len(aux_list)
n_rows = math.ceil(n_plots / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 5 * n_rows))
axes = axes.flatten()

def plot_attrition_by_category(df, col, ax):
    (
        df
        .groupby(col)['Attrition']
        .value_counts(normalize=True)
        .unstack()
        .plot(kind='bar', stacked=True, ax=ax, legend=False)
    )
    ax.set_title(col)
    ax.set_ylabel('Proportion')
    ax.tick_params(axis='x', rotation=45)
for i, col in enumerate(aux_list):
    plot_attrition_by_category(df, col, axes[i])
#plot_attrition_by_category(df, 'Gender', axes[i])
# Remove unused axes
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

# One legend for everything
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, title='Attrition', loc='upper right')

plt.tight_layout()
plt.show()


In [ ]:
df[['TotalWorkingYears','Attrition']].sort_index()

In [ ]:
df['YearsAtCompany'].plot(kind= 'hist')

In [ ]:
pairs = []
result_pairs= []
results = []

for i in numerical_cols:
    for j in numerical_cols:
        #print(i, j)
        if i != j and ((i,j) not in pairs and (j, i) not in pairs):
            pairs.append((i,j))
            aux_corr = df[[i, j]].corr().iloc[0, 1]
            if aux_corr > 0.7:
                result_pairs.append((i, j))
                results.append((df[[i, j]].corr().iloc[0, 1]))


#results_df = pd.DataFrame((sorted(results)))
#results_df.plot(kind='bar', x=pairs)
print(result_pairs)
print(len(results), len(result_pairs))

df_plot = pd.DataFrame({'value': results},index=result_pairs)

ax = df_plot.plot(kind='bar')
ax.set_title('Highest Correlated Feature Pairs')
ax.set_ylabel('Correlation Coefficient')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()


In [ ]:
OverTime
YearsAtCompany
YearsInCurrentRole
MonthlyIncome
JobLevel
Age

In [ ]:
print('MaritalStatus', df.groupby('MaritalStatus')['Attrition'].mean())
print('OverTime', df.groupby('OverTime')['Attrition'].mean())
print('BusinessTravel', df.groupby('BusinessTravel')['Attrition'].mean())
print('TravelBool', df.groupby('TravelBool')['Attrition'].mean())

In [ ]:
pd.crosstab(df['MaritalStatus'], df['Attrition'], normalize='columns')
pd.crosstab(df['BusinessTravel'], df['Attrition'], normalize='columns')

In [ ]:
print('MaritalStatus', pd.crosstab(df['MaritalStatus'], df['Attrition'], normalize='index'))
print('OverTime', pd.crosstab(df['OverTime'], df['Attrition'], normalize='index'))
print('BusinessTravel', pd.crosstab(df['BusinessTravel'], df['Attrition'], normalize='index'))
print('TravelBool', pd.crosstab(df['TravelBool'], df['Attrition'], normalize='index'))

In [ ]:
# Total count per category
total = df['MaritalStatus'].value_counts().sort_index()

# Counts split by attrition
split = pd.crosstab(df['MaritalStatus'], df['Attrition'])

# Combine into one DataFrame
plot_df = split.copy()
plot_df['Total'] = total

# Reorder columns
plot_df = plot_df[['Total', 0, 1]]  # if Attrition is 0/1

plot_df.plot(kind='bar', figsize=(10,6), color=['steelblue', 'green', 'red'])

plt.xticks(rotation=45, ha='right')
plt.ylabel("Count")
plt.title("MaritalStatus Distribution with Attrition Breakdown")
plt.tight_layout()
plt.show()

In [ ]:
# Total count per category
total = df['BusinessTravel'].value_counts().sort_index()

# Counts split by attrition
split = pd.crosstab(df['BusinessTravel'], df['Attrition'])

# Combine into one DataFrame
plot_df = split.copy()
plot_df['Total'] = total

# Reorder columns
plot_df = plot_df[['Total', 0, 1]]  # if Attrition is 0/1

plot_df.plot(kind='bar', figsize=(10,6), color=['steelblue', 'green', 'red'])

plt.xticks(rotation=45, ha='right')
plt.ylabel("Count")
plt.title("BusinessTravel Distribution with Attrition Breakdown")
plt.tight_layout()
plt.show()

In [ ]:
# Total count per category
total = df['OverTime'].value_counts().sort_index()

# Counts split by attrition
split = pd.crosstab(df['OverTime'], df['Attrition'])

# Combine into one DataFrame
plot_df = split.copy()
plot_df['Total'] = total

# Reorder columns
plot_df = plot_df[['Total', 0, 1]]  # if Attrition is 0/1

plot_df.plot(kind='bar', figsize=(10,6), color=['steelblue', 'green', 'red'])

plt.xticks(rotation=45, ha='right')
plt.ylabel("Count")
plt.title("OverTime Distribution with Attrition Breakdown")
plt.tight_layout()
plt.show()

In [ ]:
df['OverTime'] = df['OverTime'].apply(lambda x: 1 if x=='Yes' else 0)

In [ ]:
print(df[['Attrition', 'OverTime']].corr().iloc[0,1]) 

In [ ]:
df['TravelBool'] = df['BusinessTravel'].isin(['Travel_Frequently', 'Travel_Rarely']).astype(int)

In [ ]:
# Total count per category
total = df['TravelBool'].value_counts().sort_index()

# Counts split by attrition
split = pd.crosstab(df['TravelBool'], df['Attrition'])

# Combine into one DataFrame
plot_df = split.copy()
plot_df['Total'] = total

# Reorder columns
plot_df = plot_df[['Total', 0, 1]]  # if Attrition is 0/1

plot_df.plot(kind='bar', figsize=(10,6), color=['steelblue', 'green', 'red'])

plt.xticks(rotation=45, ha='right')
plt.ylabel("Count")
plt.title("TravelBool Distribution with Attrition Breakdown")
plt.tight_layout()
plt.show()

In [ ]:
# Total count per category
total = df['WorkLifeBalance'].value_counts().sort_index()

# Counts split by attrition
split = pd.crosstab(df['WorkLifeBalance'], df['Attrition'])

# Combine into one DataFrame
plot_df = split.copy()
plot_df['Total'] = total

# Reorder columns
plot_df = plot_df[['Total', 0, 1]]  # if Attrition is 0/1

plot_df.plot(kind='bar', figsize=(10,6), color=['steelblue', 'green', 'red'])

plt.xticks(rotation=45, ha='right')
plt.ylabel("Count")
plt.title("WorkLifeBalance Distribution with Attrition Breakdown")
plt.tight_layout()
plt.show()

In [ ]:
df.groupby('Attrition')['MaritalStatus'].value_counts()

In [ ]:
pos = 120 + 84 + 33
neg = 589 + 350 + 294

round(14.22222223, 2)
print('Single:')
print(round(120/pos, 2))
print(round(350/neg, 2))
print('Total:', round((120+350)/1470, 2))
print('Married:')
print(round(84/pos, 2))
print(round(589/neg, 2))
print('Total:', round((84+589)/1470, 2))
print('Divorced:')
print(round(33/pos, 2))
print(round(294/neg, 2))
print('Total:', round((33+294)/1470, 2))

In [ ]:
df.groupby('Attrition')['BusinessTravel'].value_counts()

In [ ]:
pos = 156 + 69 + 12
neg = 887 + 208 + 138

round(14.22222223, 2)
print('Travel Rarely:')
print(round(156/pos, 2))
print(round(887/neg, 2))
print('Total:', round((156+887)/1470, 2))
print('Travel Frequently:')
print(round(69/pos, 2))
print(round(208/neg, 2))
print('Total:', round((69+208)/1470, 2))
print('Non-Travel:')
print(round(12/pos, 2))
print(round(138/neg, 2))
print('Total:', round((12+138)/1470, 2))

In [ ]:
df.groupby('Attrition')['OverTime'].value_counts()

In [ ]:
pos = 127+110
neg = 944+289

print('Overtime:')
print(round(127/pos, 2))
print(round(289/neg, 2))
print('Total:', round((127+289)/1470, 2))
print('No Overtime:')
print(round(110/pos, 2))
print(round(944/neg, 2))
print('Total:', round((110+944)/1470, 2))

In [ ]:
df.head()